# Метрики качества и оценка моделей

In [ ]:
import os
import time
from collections import Counter
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import (
    KFold, GroupKFold, StratifiedKFold, TimeSeriesSplit,
    train_test_split, cross_val_score,
    GridSearchCV, RandomizedSearchCV
)
from sklearn.inspection import permutation_importance
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler, StandardScaler, PolynomialFeatures, OneHotEncoder, LabelEncoder, TargetEncoder, OrdinalEncoder
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier, plot_tree, export_graphviz
from sklearn.ensemble import (
    RandomForestRegressor, AdaBoostRegressor, StackingRegressor,
    RandomForestClassifier, AdaBoostClassifier, StackingClassifier
)
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score, f1_score,
    mean_absolute_error, root_mean_squared_error, r2_score,
    make_scorer, mean_absolute_percentage_error, roc_auc_score
)
from graphviz import Source

import shap
import optuna

from xgboost import XGBRegressor

In [ ]:
RANDOM_STATE = 21

## Базовые метрики для классификации

### Задание 1: Accuracy, Precision, Recall, F1-score

Описание:
Вы — аналитик в медицинской компании, которая разрабатывает модель для диагностики заболевания на основе анализов крови. Вам нужно оценить качество модели, предсказывающей наличие заболевания (бинарная классификация).
Требования:

Постройте простую модель (например, логистическая регрессия или дерево решений) на датасете Pima Indians Diabetes.
Рассчитайте метрики:

Accuracy
Precision
Recall
F1-score

Объясните, почему в медицинской диагностике важнее максимизировать recall, а не accuracy.
Постройте confusion matrix и проанализируйте ошибки модели.
Дополнительно:

Попробуйте изменить порог классификации и посмотрите, как это влияет на метрики.

In [ ]:
# https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database

### Задание 2: ROC-AUC и PR-AUC

Описание:
Вы работаете в банке и разрабатываете модель для обнаружения мошеннических транзакций. Данные сильно несбалансированы: мошенничество встречается в 1% случаев.
Требования:

Используйте датасет Credit Card Fraud Detection.
Постройте модель (например, случайный лес или градиентный бустинг).
Рассчитайте:

ROC-AUC
PR-AUC

Объясните, почему в данном случае PR-AUC более информативен, чем ROC-AUC.
Постройте графики ROC-кривой и PR-кривой.
Дополнительно:

Попробуйте применить методы балансировки классов (например, SMOTE) и оцените, как это повлияет на метрики.

In [ ]:
# https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud

## Метрики для регрессии

### Задание 3: MAE, MSE, RMSE, R²

Описание:
Вы аналитик в компании, которая прогнозирует цены на недвижимость. Вам нужно оценить качество модели регрессии.
Требования:

Используйте датасет Boston Housing или California Housing Prices.
Постройте модель регрессии (например, линейная регрессия, случайный лес).
Рассчитайте метрики:

MAE (Mean Absolute Error)
MSE (Mean Squared Error)
RMSE (Root Mean Squared Error)
R² (R-squared)

Объясните, в каких случаях лучше использовать RMSE, а в каких — MAE.
Проанализируйте, какие признаки сильнее всего влияют на ошибку модели.
Дополнительно:

Попробуйте логарифмировать целевую переменную и оцените, как это повлияет на метрики.


In [ ]:
# https://www.kaggle.com/datasets/vikrishnan/boston-house-prices
# https://www.kaggle.com/datasets/camnugent/california-housing-prices

## Сравнение моделей

### Задание 4: Кросс-валидация и сравнение моделей

Описание:
Вы участвуете в соревновании на Kaggle и хотите выбрать лучшую модель для задачи классификации.
Требования:

Выберите датасет (например, Titanic: Machine Learning from Disaster).
Постройте и сравните следующие модели:

Логистическая регрессия
Случайный лес
Градиентный бустинг (XGBoost/LightGBM)

Используйте кросс-валидацию (например, StratifiedKFold для классификации).
Сравните модели по метрикам:

Accuracy
F1-score
ROC-AUC

Выберите лучшую модель и объясните свой выбор.
Дополнительно:

Постройте графики обучения (learning curves) для каждой модели.


In [ ]:
# https://www.kaggle.com/c/titanic

### Задание 5: Оценка стабильности модели

Описание:
Вы разрабатываете модель для предсказания спроса на продукты в розничной сети. Вам нужно оценить стабильность модели при изменении входных данных.
Требования:

Используйте датасет Store Sales - Time Series Forecasting.
Постройте модель (например, градиентный бустинг).
Оцените стабильность модели с помощью:

Bootstrap-метода: случайным образом пересэмплируйте данные и оцените разброс метрик.
Leave-One-Out Cross-Validation (LOOCV) для небольших датасетов.

Проанализируйте, как изменяются метрики (RMSE, MAE) при изменении выборки.
Дополнительно:

Постройте доверительные интервалы для предсказаний модели.


In [ ]:
# https://www.kaggle.com/c/store-sales-time-series-forecasting

## Оценка скорости и эффективности моделей

### Задание 6: Сравнение скорости обучения и предсказания

Описание:
Вы оптимизируете модель для работы в реальном времени. Вам нужно выбрать модель, которая быстро обучается и делает предсказания.
Требования:

Используйте датасет NYC Taxi Trip Duration.
Постройте и сравните модели:

Линейная регрессия
Случайный лес
LightGBM

Оцените:

Время обучения
Время предсказания
Качество (RMSE, R²)

Выберите модель, которая обеспечивает баланс между скоростью и качеством.
Дополнительно:

Попробуйте уменьшить размерность данных с помощью PCA и оцените, как это повлияет на скорость и качество.


In [ ]:
# https://www.kaggle.com/c/nyc-taxi-trip-duration